In [ ]:
from dotenv import load_dotenv
import os
from langchain_openai import ChatOpenAI

load_dotenv()  # 加载.env文件里的变量
# print(os.getenv("DEEPSEEK_API_KEY"))  # 现在可以正常读取了

llm = ChatOpenAI(
        model="deepseek-chat",  # 使用的模型名称，目前官方推荐用 'deepseek-chat'
        api_key=os.getenv("DEEPSEEK_API_KEY"),  # 你的 DeepSeek API Key
        base_url="https://api.deepseek.com/v1",  # DeepSeek API 地址
        temperature=0,
    )

In [54]:
from typing import Optional, Union

from openai import BaseModel
from pydantic import Field
from langchain_core.output_parsers import PydanticOutputParser
from langchain_core.prompts import ChatPromptTemplate


# llm.invoke("你好")

class UserInfo(BaseModel):
    """Extracted user information, such as name,age, phone,email"""
    name:str=Field(description="The name of user")
    age:Optional[int]=Field(description="The age of user")
    email:str=Field(description="The email of user")
    phone:Optional[str]=Field(description="The phone of user")

class ConversationalResponse(BaseModel):
    response:str=Field(description="chat response from LLM")    

class FinalResponse(BaseModel):
    final_output:Union[UserInfo,ConversationalResponse]
    

parser=PydanticOutputParser(pydantic_object=FinalResponse)
prompt=ChatPromptTemplate.from_messages([
    ('system','解析用户输入并提取个人信息 {format_instructions}'),
    ('human','{query}')
])

prompt=prompt.partial(format_instructions=parser.get_format_instructions())
structured_llm=prompt | llm | parser



In [56]:
structured_llm.invoke("我叫奥特曼，今年38岁，邮箱地址是aoteman@qq.com,电话是123123123")

FinalResponse(final_output=UserInfo(name='奥特曼', age=38, email='aoteman@qq.com', phone='123123123'))

In [ ]:
from tkinter import END
from typing import Optional

from langgraph.graph import START, StateGraph
from pydantic import BaseModel, Field
from typing_extensions import TypedDict
from IPython.display import display,Image

class State(TypedDict):
    question:str
    response:list[str]
    

class UserInfo(BaseModel):
    """Extracted user information, such as name,age, phone,email"""
    name:str=Field(description="The name of user")
    age:Optional[int]=Field(description="The age of user")
    email:str=Field(description="The email of user")
    phone:Optional[str]=Field(description="The phone of user")
    
def chat_with_model(state):
    messages=state['question']
    response=llm.invoke(messages)
    return {'response':[response.content]}
    
def insert_db(state):
    print('insert_db',state)
    
def print_result(state):
    print('print_result',state['response'])
    
def routing_function(state):
    if '我是' in state['response'][-1]:
        return 'insert_db'
    else:
        return 'print_result'
    
builder=StateGraph(State)

builder.add_node('chat_with_model',chat_with_model)
builder.add_node('insert_db',insert_db)
builder.add_node('print_result',print_result)

builder.add_edge(START,'chat_with_model')

builder.add_conditional_edges('chat_with_model',routing_function,
                              {
                                  'insert_db':'insert_db',
                                  'print_result':'print_result'
                              })

graph=builder.compile()



In [ ]:
display(Image(graph.get_graph().draw_mermaid_png()))

In [ ]:
graph.invoke({'question':'你好，请介绍一下你自己'})

In [ ]:
from typing import Optional, Union

from langgraph.graph import START, StateGraph
from pydantic import BaseModel, Field
from typing_extensions import TypedDict
from IPython.display import display,Image
from langchain_core.output_parsers import PydanticOutputParser
from langchain_core.prompts import ChatPromptTemplate



class State(TypedDict):
    question:str
    response:list[str]
    

class UserInfo(BaseModel):
    """Extracted user information, such as name,age, phone,email"""
    name:str=Field(description="The name of user")
    age:Optional[int]=Field(description="The age of user")
    email:str=Field(description="The email of user")
    phone:Optional[str]=Field(description="The phone of user")

class ConversationalResponse(BaseModel):
    """Respond to the user's query in a conversational manner. Be kind and helpful."""
    response:str=Field(description="A conversational response to the user's query")
    
class FinalResponse(BaseModel):
    final_output:Union[UserInfo,ConversationalResponse]

# structured_llm=llm.with_structured_output(FinalResponse)

parser= PydanticOutputParser(pydantic_object=FinalResponse)
prompt=ChatPromptTemplate.from_messages([
    ('system','解析用户输入并提取个人信息. {format_instructions}'),
    ('human',"{query}")
])

prompt=prompt.partial(format_instructions=parser.get_format_instructions())
structured_llm=prompt | llm | parser

def chat_with_model(state):
    messages=state['question']
    response=structured_llm.invoke(messages)
    return {'response':[response]}
    
def insert_db(state):
    print('insert_db',state)
    
def print_result(state):
    print('print_result',state['response'])
    
def routing_function(state):
    if isinstance(state['response'][-1].final_output, UserInfo):
        return 'insert_db'
    else:
        return 'print_result'
    
builder=StateGraph(State)

builder.add_node('chat_with_model',chat_with_model)
builder.add_node('insert_db',insert_db)
builder.add_node('print_result',print_result)

builder.add_edge(START,'chat_with_model')

builder.add_conditional_edges('chat_with_model',routing_function,
                              {
                                  'insert_db':'insert_db',
                                  'print_result':'print_result'
                              })

graph=builder.compile()



In [ ]:
from IPython.display import Image,display
display(Image(graph.get_graph(xray=True).draw_mermaid_png()))

In [ ]:
extracted_user_info=structured_llm.invoke("你好")

extracted_user_info


In [ ]:
graph.invoke({'question':'我叫奥特曼，今年38岁，邮箱地址是aoteman@qq.com,电话是123123123'})

In [ ]:
graph.invoke({'question':'你好'})